Random Forest Classifier

In [2]:
from sklearn.ensemble import RandomForestClassifier
import pickle

#los datos train y test están en train.csv y test.csv, los cargamos

import pandas as pd
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
X_train = train.drop(columns=["Target"])
y_train = train["Target"]

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

with open("model.pkl", "wb") as f:
    pickle.dump(clf, f)


SVM

In [ ]:
from sklearn.svm import SVC
import pickle
import pandas as pd

# cargar datasets
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

X_train = train.drop(columns=["Target"])
y_train = train["Target"]

# modelo SVM
clf = SVC(
    kernel="rbf",
    probability=True,   # necesario si luego usas predict_proba
    random_state=42
)

clf.fit(X_train, y_train)

# guardar modelo
with open("model_SVM.pkl", "wb") as f:
    pickle.dump(clf, f)

Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import pickle

#los datos train y test están en train.csv y test.csv, los cargamos

import pandas as pd
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
X_train = train.drop(columns=["Target"])
y_train = train["Target"]

clf = DecisionTreeClassifier(random_state=42, max_depth=5)
clf.fit(X_train, y_train)

with open("model_tree.pkl", "wb") as f:
    pickle.dump(clf, f)


Red neuronal

In [1]:
# Hacemos lo mismo que antes usando una red neuronal de pytorch
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
# Cargar datos
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
X_train = train.drop(columns=["Target"])
y_train = train["Target"]
# Normalizar características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
# Convertir a tensores
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
# Definir modelo de red neuronal
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out
input_size = X_train_tensor.shape[1]
hidden_size = 64
num_classes = len(y_train.unique())
model = SimpleNN(input_size, hidden_size, num_classes)
# Entrenar modelo
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
# Guardar modelo
# torch.save(model.state_dict(), "model_nn.pth")
torch.save(model, "model_nn.pth")

#Vemos su accuracy en el test
X_test = test.drop(columns=["Target"])
y_test = test["Target"]
X_test_scaled = scaler.transform(X_test)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
model.eval()
with torch.no_grad():
    outputs = model(X_test_tensor)
    _, predicted = torch.max(outputs.data, 1)
accuracy = (predicted == torch.tensor(y_test.values)).sum().item() / len(y_test)
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.8300


Regresión

In [1]:
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import pickle


def crear_modelo_regresion():
    # 1. Cargar el dataset de Diabetes (Regresión)
    print("Cargando dataset de Diabetes...")
    data = load_diabetes(as_frame=True)
    df = data.frame
    
    # 2. Dividir en Train y Test (80% / 20%)
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
    
    # 3. Guardar en archivos CSV
    train_df.to_csv('train_regression.csv', index=False)
    test_df.to_csv('test_regression.csv', index=False)
    print("Archivos 'train_regression.csv' y 'test_regression.csv' guardados exitosamente.")
    
    # 4. Separar características (X) y variable objetivo (y)
    X_train = train_df.drop(columns=['target'])
    y_train = train_df['target']
    X_test = test_df.drop(columns=['target'])
    y_test = test_df['target']
    
    # 5. Entrenar un modelo de Regresión Lineal pequeño
    print("Entrenando el modelo de Regresión Lineal...")
    modelo = LinearRegression()
    modelo.fit(X_train, y_train)
    
    # 6. Evaluar rápidamente
    predicciones = modelo.predict(X_test)
    mse = mean_squared_error(y_test, predicciones)
    r2 = r2_score(y_test, predicciones)
    
    print(f"Modelo entrenado. MSE: {mse:.2f} | R2 Score: {r2:.2f}\n")
    return modelo

if __name__ == "__main__":
    modelo_reg = crear_modelo_regresion()
    with open("model_regression.pkl", "wb") as f:
        pickle.dump(modelo_reg, f)

Cargando dataset de Diabetes...
Archivos 'train_regression.csv' y 'test_regression.csv' guardados exitosamente.
Entrenando el modelo de Regresión Lineal...
Modelo entrenado. MSE: 2900.19 | R2 Score: 0.45



Multiclase

In [2]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle


def crear_modelo_multiclase():
    # 1. Cargar el dataset Wine (Clasificación Multiclase: 3 clases)
    print("Cargando dataset Wine...")
    data = load_wine(as_frame=True)
    df = data.frame
    
    # 2. Dividir en Train y Test (80% / 20%)
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])
    
    # 3. Guardar en archivos CSV
    train_df.to_csv('train_multiclass.csv', index=False)
    test_df.to_csv('test_multiclass.csv', index=False)
    print("Archivos 'train_multiclass.csv' y 'test_multiclass.csv' guardados exitosamente.")
    
    # 4. Separar características (X) y variable objetivo (y)
    X_train = train_df.drop(columns=['target'])
    y_train = train_df['target']
    X_test = test_df.drop(columns=['target'])
    y_test = test_df['target']
    
    # 5. Entrenar un modelo RandomForest pequeño
    print("Entrenando el modelo RandomForestClassifier...")
    modelo = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
    modelo.fit(X_train, y_train)
    
    # 6. Evaluar rápidamente
    predicciones = modelo.predict(X_test)
    acc = accuracy_score(y_test, predicciones)
    
    print(f"Modelo entrenado. Accuracy: {acc:.2f}")
    print("Reporte de clasificación:\n", classification_report(y_test, predicciones))
    
    return modelo

if __name__ == "__main__":
    modelo_multi = crear_modelo_multiclase()
    with open("model_multiclass.pkl", "wb") as f:
        pickle.dump(modelo_multi, f)   

Cargando dataset Wine...
Archivos 'train_multiclass.csv' y 'test_multiclass.csv' guardados exitosamente.
Entrenando el modelo RandomForestClassifier...
Modelo entrenado. Accuracy: 1.00
Reporte de clasificación:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36

